In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import urllib.request
import zipfile
import tempfile
from shapely.geometry import Polygon, MultiPoint, Point, MultiPolygon
from shapely import from_wkt
import geopandas as gpd
import h3
import folium
import dask
import dask.dataframe as dd
from dask.diagnostics import ProgressBar

import swifter
import pandas as pd
import numpy as np
import pyproj
from tqdm import tqdm

import osmnx as ox

from scipy.spatial import distance_matrix
 
dask.config.set(scheduler='processes')

# 1.  Extract Coordinates and Define Bounding Box

### 1.1. Extract coordinates

In [ ]:
# Define a function to extract x and y from 'location' column
def extract_coordinates(df):
    df['x'] = df['location'].str.extract(r'POINT \(([^ ]+) [^ ]+\)').astype(float)
    df['y'] = df['location'].str.extract(r'POINT \([^ ]+ ([^ ]+)\)').astype(float)
    return df

In [ ]:
# Define a function to convert point into latlong
def convert_projection_to_latlon(x):
    # Define the target projection (latitude and longitude)
    target_proj = pyproj.Proj(proj='latlong', datum='WGS84')

    # Convert the coordinates
    source_proj = pyproj.Proj(proj='utm', zone=10, datum='WGS84')
    lon, lat = pyproj.transform(source_proj, target_proj, x[3], x[4])
    return lat, lon

In [ ]:
def rev(list):
    # Reverse internal elements of list
    new_list = []
    for item in list:
        new_list.append(item[::-1])
    return new_list

### 1.2. Define bounding box

In [ ]:
def sf_dataframe_and_bbox():
    sf_df = pd.read_csv(file_path, sep='\t')
    sf_df = extract_coordinates(sf_df)
    min_x = sf_df['x'].min()
    max_x = sf_df['x'].max()
    min_y = sf_df['y'].min()
    max_y = sf_df['y'].max()
    # Print the overall bounding box
    print(f"Bounding box: ({min_x}, {min_y}, {max_x}, {max_y})")

    return sf_df, min_x, max_x, min_y, max_y

file_path = './San_francisco_1k_15months_traj.tsv'
sf_df, min_x, max_x, min_y, max_y = sf_dataframe_and_bbox()

# 2. Hexagon Generation

### 2.1. Create hexagons by resolution

In [ ]:
def create_sf_hex_map(min_x, min_y, max_x, max_y, res_1, res_2):
    url = "https://data.cdc.gov/download/n44h-hy2j/application/zip"

    with tempfile.TemporaryDirectory() as tmpdir:
        zip_path = os.path.join(tmpdir, "counties.zip")
        urllib.request.urlretrieve(url, zip_path)

        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(tmpdir)

        shp_file = [f for f in os.listdir(tmpdir) if f.endswith('.shp')][0]
        shp_path = os.path.join(tmpdir, shp_file)

        # Read and process SF boundary
        counties = gpd.read_file(shp_path)
        sf_county = counties[counties['NAME'] == 'San Francisco'].copy()
        sf_county = sf_county.to_crs(epsg=4326)

        # Buffer the geometry slightly in degrees to catch edge hexagons and complete coverage 
        buffered_geometry = sf_county.geometry.buffer(0.005)

        # Create a map
        m = folium.Map(
            location=[37.7749, -122.4194],
            zoom_start=12,
            tiles='OpenStreetMap'
        )

        # Add original SF county boundary
        folium.GeoJson(
            sf_county,
            style_function=lambda x: {
                'color': 'red',
                'weight': 2,
                'fillOpacity': 0
            },
            name="San Francisco County"
        ).add_to(m)

        # Iterate over all polygons in the MultiPolygon (in case there are multiple disconnected parts)
        all_hexagons_1 = set()
        all_hexagons_2 = set()
        for geometry in buffered_geometry:
            if geometry.geom_type == 'MultiPolygon':
                polygons = list(geometry.geoms)
            else:
                polygons = [geometry]

            for polygon in polygons:

                # Create GeoJSON format for polyfill
                geojson = {
                        "type": "Polygon",
                        "coordinates": [list(polygon.exterior.coords)]
                          }
                # Get hexagons at specified resolutions
                # Using h3 version 3.*
                # hexagons_1 = h3.polyfill(geojson, res_1, geo_json_conformant=True)
                # hexagons_2 = h3.polyfill(geojson, res_2, geo_json_conformant=True)
                
                # Using h3 version 4.*
                hexagons_1 = h3.geo_to_cells(geojson, res_1)
                hexagons_2 = h3.geo_to_cells(geojson, res_2)
                    
                all_hexagons_1.update(hexagons_1)
                all_hexagons_2.update(hexagons_2)
        
        # Using h3 version 3.*
        # hexagons_1_gdf = gpd.GeoDataFrame({
        #     'h3_index': list(all_hexagons_1),
        #     'geometry': [Polygon(rev(h3.h3_to_geo_boundary(h))) for h in all_hexagons_1]
        # }, crs='EPSG:4326')

        # hexagons_2_gdf = gpd.GeoDataFrame({
        #     'h3_index': list(all_hexagons_2),
        #     'geometry': [Polygon(rev(h3.h3_to_geo_boundary(h))) for h in all_hexagons_2]
        # }, crs='EPSG:4326')
        

        '''  Uncomment this section (lines 76 - 85 if you use h3 version 4.* and comment lines 63 - 72 above) '''   
        # Create GeoDataFrames of hexagons
        hexagons_1_gdf = gpd.GeoDataFrame({
            'h3_index': list(all_hexagons_1),
            'geometry': [Polygon(rev(h3.cell_to_boundary(h))) for h in all_hexagons_1]
        }, crs='EPSG:4326')

        hexagons_2_gdf = gpd.GeoDataFrame({
            'h3_index': list(all_hexagons_2),
            'geometry': [Polygon(rev(h3.cell_to_boundary(h))) for h in all_hexagons_2]
        }, crs='EPSG:4326')
        

        # Filter hexagons to only those that intersect with the SF boundary
        intersecting_hexagons_1 = hexagons_1_gdf[hexagons_1_gdf.intersects(sf_county.unary_union)]
        intersecting_hexagons_2 = hexagons_2_gdf[hexagons_2_gdf.intersects(sf_county.unary_union)]

        ''' Uncomment lines 93 - 95 if you use h3 version 4.* and comment lines 88 - 90 above '''
        # Filter hexagons to only those that intersect with the SF boundary
        # intersecting_hexagons_1 = hexagons_1_gdf[hexagons_1_gdf.intersects(sf_county.union_all())]
        # intersecting_hexagons_2 = hexagons_2_gdf[hexagons_2_gdf.intersects(sf_county.union_all())]
        
        
        # Add larger resolution hexagons (red)
        for _, row in intersecting_hexagons_1.iterrows():
            folium.Polygon(
                locations=[[b[1], b[0]] for b in row['geometry'].exterior.coords],
                color='red',
                weight=2,
                fill=False,
                fill_opacity=0.1
            ).add_to(m)

        # Add smaller resolution hexagons (black)
        for _, row in intersecting_hexagons_2.iterrows():
            folium.Polygon(
                locations=[[b[1], b[0]] for b in row['geometry'].exterior.coords],
                color='black',
                weight=1,
                fill=False,
                fill_opacity=0.1
            ).add_to(m)

        print(f"Number of resolution {res_1} hexagons: {len(intersecting_hexagons_1)}")
        print(f"Number of resolution {res_2} hexagons: {len(intersecting_hexagons_2)}")

        return m, sf_county, intersecting_hexagons_1, intersecting_hexagons_2
    
    
map_view, sf_boundary, hex_res1, hex_res2 = create_sf_hex_map(min_x=min_x, min_y=min_y, max_x=max_x, max_y=max_y, res_1=8, res_2=8)

### 2.2. Visualize trajectories in hexagons

In [ ]:
# This function takes DataFrame with all points with (x,y), agent Id (traj id), and DataFrame that contain hex_Id and hex Polygon
def traj_in_sub_region(sf_df, agentId, hex_res1):
    
    # Convert traj coords into latlong and making array representation of trajectory
    sub_df = sf_df.loc[sf_df["agentId"] == agentId].drop_duplicates(subset='location', keep='first').sort_values(by="simulationTime")
    df = sub_df.swifter.apply(lambda x: convert_projection_to_latlon(x),axis=1)
    traj = np.asarray(df)
    
    # Exclude hexes where the is no traj occurance
    hex_along_traj = hex_res1[hex_res1.intersects(MultiPoint(rev(traj)))]

    # Extract information for each point, in which hex it is falling
    data = []
    for point in rev(traj):
        data.append(np.asarray([hex_along_traj[hex_along_traj.intersects(Point(point))]['h3_index'].values[0],Point(point)]))
    hex_point_df = pd.DataFrame(data,columns=['hex_id','point'])
    
    # Extracting only points that have a neighbour in same hex (previous or next) 
    index_list = []
    for index, point in hex_point_df.iterrows():
        if index == 0:
            if data[index][0] != data[index+1][0]:
                index_list.append(index)
            continue
        if index+1 == len(hex_point_df):
            if data[index][0] != data[index-1][0]:
                index_list.append(index)
            continue
        if data[index][0] != data[index+1][0] and data[index][0] != data[index-1][0]:
            index_list.append(index)
    hex_point_df.drop(index=index_list, inplace=True)
    
    # After all, combine all points of sub-traj in Multipoint that falles in hex and make DataFrame with ["hex_id", "Multipoint"]
    data = []
    for point in hex_point_df['hex_id'].unique():
        data.append(np.asarray([point,MultiPoint(hex_point_df.loc[hex_point_df['hex_id']==point]['point'].values)]))
    hex_multi_df = pd.DataFrame(data,columns=['hex_id','Multi_point'])
    hex_along_traj = hex_along_traj.loc[hex_along_traj['h3_index'].isin(hex_multi_df['hex_id'])]

    # visual
    for _, row in hex_multi_df.iterrows():
        folium.PolyLine(
            locations=[[b.coords[0][1], b.coords[0][0]] for b in row['Multi_point'].geoms],
            color="blue",
            weight=1,
            opacity=0.7
        ).add_to(map_view)
    for _, row in hex_along_traj.iterrows():
                folium.Polygon(
                    locations=[[b[1], b[0]] for b in row['geometry'].exterior.coords],
                    color='blue',
                    weight=5,
                    fill=False,
                    fill_opacity=0.1,
                    tooltip=row['h3_index'],
                ).add_to(map_view)
    return hex_multi_df

In [ ]:
# map_view

In [ ]:
# Going through all traj dividing it into sub-traj and storing with hex that they fall into
global_hex_multi_df = pd.DataFrame(columns=['hex_id','Multi_point'])
for agent in tqdm(sf_df['agentId'].unique()):
    global_hex_multi_df = pd.concat([global_hex_multi_df,traj_in_sub_region(sf_df=sf_df,agentId=agent,hex_res1=hex_res1)])
global_hex_multi_df.reset_index(drop=True,inplace=True)

# 3. Generate Hexagon Features for a rich Dataset

### 3.1. Feature s1: Types of amenities per hexagon

In [ ]:
# Function to extract types of amenities in each hex
def amenities_types_in_hex(global_hex_multi_df):
    amenities_types = []
    for hex_id in tqdm(global_hex_multi_df['hex_id'].unique()):
        osm_amen = ox.features.features_from_polygon(
            hex_res1.loc[hex_res1['h3_index'] == hex_id]['geometry'].values[0], 
            tags={"amenity": True}
        )
        amenity_types = osm_amen['amenity'].unique()
        amenities_types.append([hex_id, amenity_types])
    hex_amen_types_df = pd.DataFrame(amenities_types, columns=['hex_id', 'amenity_types'])
    return hex_amen_types_df  # s1


# hex_amen_types_df = amenities_types_in_hex(global_hex_multi_df)

### 3.2. Feature s2: Number of amenities per hexagon

In [ ]:
# Take all Amenities in hex and give our DataFrame [hex_id, Amenities Count, Unique Amenities Count, Most common Amenity]
def amenities_count_in_hex(global_hex_multi_df):
    amenities_count = []
    for hex in tqdm(global_hex_multi_df['hex_id'].unique()):
        osm_amen = ox.features.features_from_polygon(hex_res1.loc[hex_res1['h3_index']==hex]['geometry'].values[0], tags={"amenity": True})
        count = []
        for amen in osm_amen['amenity'].unique():
            count.append(osm_amen.loc[osm_amen['amenity']==amen]['amenity'].count())
        amenities_count.append([hex, len(osm_amen),len(osm_amen['amenity'].unique()),osm_amen['amenity'].unique()[np.asarray(count).argmax()]])
    hex_amen_count_df = pd.DataFrame(amenities_count,columns=['hex_id','amen_count','amen_uniq_count','most_common_amen'])  
    return hex_amen_count_df  # s2


# hex_amen_df = amenities_count_in_hex(global_hex_multi_df)

### 3.3. Feature s3: Number of sub-trajectories per hexagon

In [ ]:
# Function to count the number of sub-trajectories in each hex
def sub_trajectory_count_in_hex(global_hex_multi_df):
    sub_traj_count_df = global_hex_multi_df.groupby('hex_id').size().reset_index(name='sub_traj_count')
    return sub_traj_count_df  # s3


# sub_traj_count_df = sub_trajectory_count_in_hex(global_hex_multi_df)

### 3.4. Feature s4: Total trajectory length per hexagon

In [ ]:
from geopy.distance import great_circle

#  Function to calculate total length of trajectories for each hex using great-circle distance.
def total_traj_length_in_hex(global_hex_multi_df):
    hex_dist = []
    for hex_id in tqdm(global_hex_multi_df['hex_id'].unique()):
        total_dist = 0  # Total distance in meters for the current hex
        hex_rows = global_hex_multi_df[global_hex_multi_df['hex_id'] == hex_id]
        for _, hex_row in hex_rows.iterrows():
            sub_points = []
            for point in hex_row['Multi_point'].geoms:
                # Append (latitude, longitude) tuple
                sub_points.append((point.y, point.x))  # Note: geopy uses (lat, lon)
            # Calculate distance for the sub-trajectory
            dist = 0  # Distance for the current sub-trajectory
            for traj_idx in range(len(sub_points) - 1):
                # Calculate great-circle distance between consecutive points
                dist += great_circle(sub_points[traj_idx], sub_points[traj_idx + 1]).meters
            total_dist += dist  # Add sub-trajectory distance to total
        hex_dist.append([hex_id, total_dist])
    hex_dist_df = pd.DataFrame(hex_dist, columns=['hex_id', 'total_traj_length'])
    return hex_dist_df  # s4


# hex_dist_df = total_traj_length_in_hex(global_hex_multi_df)

### 3.5.1. Feature s5a: Mean sub-trajectory centroid to hexagon centroid distance

In [ ]:
def mean_sub_traj_centroid_to_hex_centroid_distance(global_hex_multi_df, hex_res1):
    """
    Compute mean great-circle distance from sub-trajectory centroids to hex centroid.
    Returns a DataFrame with 'hex_id' and 'mean_sub_traj_centroid_distance'.
    """
    distance_list = []
    for hex_id in tqdm(global_hex_multi_df['hex_id'].unique()):
        # Get hexagon geometry and centroid
        hex_geom_row = hex_res1.loc[hex_res1['h3_index'] == hex_id]
        if hex_geom_row.empty:
            continue  
        hex_geom = hex_geom_row['geometry'].values[0]
        hex_centroid = hex_geom.centroid
        hex_centroid_coords = (hex_centroid.y, hex_centroid.x)  # (latitude, longitude)
        # Get sub-trajectories in this hex
        sub_traj_df = global_hex_multi_df[global_hex_multi_df['hex_id'] == hex_id]
        distances = []
        for _, row in sub_traj_df.iterrows():
            sub_traj_centroid = row['Multi_point'].centroid
            sub_traj_centroid_coords = (sub_traj_centroid.y, sub_traj_centroid.x)
            # Calculate great-circle distance in meters
            distance = great_circle(hex_centroid_coords, sub_traj_centroid_coords).meters
            distances.append(distance)
        if distances:
            mean_distance = np.mean(distances)
        else:
            mean_distance = np.nan  # Handle hexes with no sub-trajectories
        distance_list.append([hex_id, mean_distance])
    mean_distance_df = pd.DataFrame(distance_list, columns=['hex_id', 'mean_sub_traj_centroid_distance'])
    return mean_distance_df  # s5


# mean_sub_traj_cent_to_hex_cent_df = mean_sub_traj_centroid_to_hex_centroid_distance(global_hex_multi_df, hex_res1)

### 3.5.2. Feature s5b: Mean point to hexagon centroid distance

In [ ]:
def mean_point_to_hex_centroid_distance(global_hex_multi_df, hex_res1):
    """
    Compute mean great-circle distance from each point to the hex centroid.
    Returns a DataFrame with 'hex_id' and 'mean_point_distance'.
    """
    distance_list = []
    for hex_id in tqdm(global_hex_multi_df['hex_id'].unique()):
        # Get hexagon geometry and centroid
        hex_geom_row = hex_res1.loc[hex_res1['h3_index'] == hex_id]
        if hex_geom_row.empty:
            continue 
        hex_geom = hex_geom_row['geometry'].values[0]
        hex_centroid = hex_geom.centroid
        hex_centroid_coords = (hex_centroid.y, hex_centroid.x)  # (latitude, longitude)
        # Get sub-trajectories in this hex
        sub_traj_df = global_hex_multi_df[global_hex_multi_df['hex_id'] == hex_id]
        distances = []
        for _, row in sub_traj_df.iterrows():
            for point in row['Multi_point'].geoms:
                point_coords = (point.y, point.x)  # (latitude, longitude)
                # Calculate great-circle distance in meters
                distance = great_circle(hex_centroid_coords, point_coords).meters
                distances.append(distance)
        if distances:
            mean_distance = np.mean(distances)
        else:
            mean_distance = np.nan  # Handle hexes with no points
        distance_list.append([hex_id, mean_distance])
    mean_distance_df = pd.DataFrame(distance_list, columns=['hex_id', 'mean_point_distance'])
    return mean_distance_df  # s5b


# mean_points_to_hex_cent_df = mean_point_to_hex_centroid_distance(global_hex_multi_df, hex_res1)

### 3.6. Feature s6a and s6b: Water area percentage and Tree area percentage per hexagon

In [ ]:
# Calculate percentage of water and tree areas in all hexes
def area_percentage_in_hex(global_hex_multi_df, hex_res1):
    hex_water_area_data = []
    hex_tree_area_data = []
    for hex_id in tqdm(global_hex_multi_df['hex_id'].unique()):
        hex_geom = hex_res1.loc[hex_res1['h3_index'] == hex_id]['geometry'].values[0]
        hex_area = hex_geom.area
        water_area = 0
        tree_area = 0
        # Calculate water area
        try:
            osm_water = ox.features.features_from_polygon(
                hex_geom, tags={"natural": ["water", "bay"]}
            )
            for water_geom in osm_water['geometry']:
                water_area += water_geom.intersection(hex_geom).area
        except:
            pass
        # Calculate tree area
        try:
            osm_trees = ox.features.features_from_polygon(
                hex_geom,
                tags={"landuse": ["forest", "grass"], "natural": "tree"}
            )
            for tree_geom in osm_trees['geometry']:
                tree_area += tree_geom.intersection(hex_geom).area
        except:
            pass
        water_area_pct = (water_area / hex_area) * 100 if water_area > 0 else 0
        tree_area_pct = (tree_area / hex_area) * 100 if tree_area > 0 else 0
        hex_water_area_data.append([hex_id, water_area_pct])
        hex_tree_area_data.append([hex_id, tree_area_pct])
    hex_water_area_df = pd.DataFrame(hex_water_area_data, columns=['hex_id', 'water_area_%'])
    hex_tree_area_df = pd.DataFrame(hex_tree_area_data, columns=['hex_id', 'tree_area_%'])
    return hex_water_area_df, hex_tree_area_df  # s6a and s6b


# hex_water_area_df,  hex_tree_area_df = area_percentage_in_hex(global_hex_multi_df,hex_res1)

### 3.7. Feature s7: Amenities along sub-trajectories per hexagon (Separate File)

In [ ]:
import ast
from collections import Counter

def total_amenities_along_traj(amenities_string):
    amenities_list = ast.literal_eval(amenities_string)
    return sum(count for _, count in amenities_list)

amenities_df = pd.read_csv('./csv_files/Amenities_count_along_traj_s7.csv')

# Count total amenities for each hex_id
amenities_df['amen_along_trajs'] = amenities_df['amen_count_traj'].apply(total_amenities_along_traj)

s7_df = amenities_df.sort_values('amen_along_trajs', ascending=False)
s7_df.head(2)

# 4. Compute Pairwise Hexagon Similarity Scores 

### 4.1. Categorical amenities (Jaccard Similarity) and numerical similarity measures (s2 to s7)

In [ ]:
#  Compute the Jaccard similarity matrix for categorical data using .loc to avoid IndexError
def compute_jaccard_similarity(df, column_name, hex_ids_list):
    data_dict = dict(zip(df['hex_id'], df[column_name]))
    sim_matrix = pd.DataFrame(index=hex_ids_list, columns=hex_ids_list, dtype=float)
    
    for hex_id_i in tqdm(hex_ids_list):
        set_i = set(data_dict.get(hex_id_i, []))
        for hex_id_j in hex_ids_list:
            set_j = set(data_dict.get(hex_id_j, []))
            union = set_i | set_j
            intersection = set_i & set_j
            if union:
                similarity = len(intersection) / len(union)
            else:
                similarity = 1  # Both sets are empty
            sim_matrix.loc[hex_id_i, hex_id_j] = similarity
    return sim_matrix

#  Compute the similarity matrix for numerical data using .loc to avoid IndexError
def compute_numerical_similarity(df, column_name, hex_ids_list):
    values = df.set_index('hex_id')[column_name]
    min_value, max_value = values.min(), values.max()
    sim_matrix = pd.DataFrame(index=hex_ids_list, columns=hex_ids_list, dtype=float)
    
    for hex_id_i in tqdm(hex_ids_list):
        val_i = values.get(hex_id_i, 0)
        for hex_id_j in hex_ids_list:
            val_j = values.get(hex_id_j, 0)
            if max_value != min_value:
                similarity = 1 - abs(val_i - val_j) / (max_value - min_value)
            else:
                similarity = 1
            sim_matrix.loc[hex_id_i, hex_id_j] = similarity
    return sim_matrix


In [ ]:
def compute_similarity_scores(s1_df, s2_df, s3_df, s4_df, s5_df, s5b_df, s6a_df, s6b_df, s7_df, s8_df):
    """
    Compute pairwise similarity scores for S1 to S6b using consistent hex_id lists.
    Returns a dictionary of similarity matrices.
    """
    # Get the union of all hex_ids from the DataFrames
    hex_ids_set = set(s1_df['hex_id']).union(
        s2_df['hex_id'], s3_df['hex_id'], s4_df['hex_id'],
        s5_df['hex_id'], s5b_df['hex_id'], s6a_df['hex_id'],
        s6b_df['hex_id'], s7_df['hex_id'], s8_df['hex_id']
    )
    hex_ids_list = list(hex_ids_set)
    
    similarity_matrices = {}
    
    # --- S1,S7,S8: Categorical Amenities (Jaccard Similarity) ---
    print("Computing similarity matrix for S1 (Categorical Amenities)...")
    s1_matrix = compute_jaccard_similarity(s1_df, 'amenity_types', hex_ids_list)
    similarity_matrices['s1'] = s1_matrix

    # --- Numerical Similarity Measures (S2 to S6b) ---
    numerical_similarity_scores = {
        's2': (s2_df, 'amen_count'),
        's3': (s3_df, 'sub_traj_count'),
        's4': (s4_df, 'total_traj_length'),
        's5': (s5_df, 'mean_sub_traj_centroid_distance'),
        's5b': (s5b_df, 'mean_point_distance'),
        's6a': (s6a_df, 'water_area_%'),
        's6b': (s6b_df, 'tree_area_%')
    }
    
    for score_name, (df, column_name) in numerical_similarity_scores.items():
        print(f"Computing similarity matrix for {score_name.upper()} ({column_name})...")
        sim_matrix = compute_numerical_similarity(df, column_name, hex_ids_list)
        similarity_matrices[score_name] = sim_matrix
    
    s7_matrix = compute_jaccard_similarity(s7_df, 'amen_along_traj', hex_ids_list)
    s8_matrix = compute_jaccard_similarity(s8_df, 'hex_geohash', hex_ids_list)
    similarity_matrices['s7'] = s7_matrix
    similarity_matrices['s8'] = s8_matrix

    return similarity_matrices


### 4.2. Compute mean similarity score for s1 to s7

In [ ]:
def compute_mean_similarity(similarity_matrices):
    """
    Compute the mean similarity score from S1 to S7.
    Returns a DataFrame containing the overall similarity between hexagons.
    """
    # Assuming all similarity matrices have the same index and columns
    hex_ids = similarity_matrices['s1'].index
    mean_similarity_matrix = pd.DataFrame(index=hex_ids, columns=hex_ids, dtype=float)
    
    print("Computing mean similarity matrix from S1 to S8...")
    for hex_id_i in tqdm(hex_ids):
        for hex_id_j in hex_ids:
            total_similarity = 0
            count = 0
            for sim_name, sim_matrix in similarity_matrices.items():
                sim_value = sim_matrix.loc[hex_id_i, hex_id_j]
                if pd.notna(sim_value):
                    total_similarity += float(sim_value)
                    count += 1
            mean_similarity = total_similarity / count if count > 0 else 0
            mean_similarity_matrix.loc[hex_id_i, hex_id_j] = mean_similarity
    return mean_similarity_matrix


### 4.3. Save similarity scores to csv files for further computations

In [ ]:
# TODO Modify such that s7 is included here
def compute_and_save_similarity_scores(global_hex_multi_df, hex_res1):
    # s1: Categorical Amenities (types of amenities)
    s1 = amenities_types_in_hex(global_hex_multi_df)
    # Save to CSV, dropping zeros
    s1_nonzero = s1[s1['amenity_types'].apply(lambda x: len(x) > 0)].copy()
    # Sort amenity_types alphabetically and convert to strings
    s1_nonzero['amenity_types'] = s1_nonzero['amenity_types'].apply(lambda x: ','.join(sorted(x)))
    s1_nonzero.to_csv('./csv_files/amenities_type_s1.csv', index=False)
    print("Similarity Score s1 (Categorical Amenities) saved to 'amenities_type_s1.csv'")
    
    # s2: Total count of Amenities
    s2_full = amenities_count_in_hex(global_hex_multi_df)
    s2 = s2_full[['hex_id', 'amen_count']]
    # Save to CSV, dropping zeros
    s2_nonzero = s2[s2['amen_count'] > 0].copy()
    s2_nonzero = s2_nonzero.sort_values(by='amen_count', ascending=False)
    s2_nonzero.to_csv('./csv_files/amenities_count_s2.csv', index=False)
    print("\nSimilarity Score s2 (Total Count of Amenities) saved to 'amenities_count_s2.csv'")
    
    # s3: Number of sub-trajectories in a hex
    s3 = sub_trajectory_count_in_hex(global_hex_multi_df)
    # Fill NaN with 0
    s3['sub_traj_count'] = s3['sub_traj_count'].fillna(0)
    # Save to CSV, dropping zeros
    s3_nonzero = s3[s3['sub_traj_count'] > 0].copy()
    s3_nonzero = s3_nonzero.sort_values(by='sub_traj_count', ascending=False)
    s3_nonzero.to_csv('./csv_files/sub_traj_count_s3.csv', index=False)
    print("\nSimilarity Score s3 (Number of Sub-trajectories in a Hex) saved to 'sub_traj_count_s3.csv'")
    
    # s4: Total length of all trajectories in a hex
    s4 = total_traj_length_in_hex(global_hex_multi_df)
    # Save to CSV, dropping zeros
    s4_nonzero = s4[s4['total_traj_length'] > 0].copy()
    s4_nonzero = s4_nonzero.sort_values(by='total_traj_length', ascending=False)
    s4_nonzero.to_csv('./csv_files/total_traj_length_s4.csv', index=False)
    print("\nSimilarity Score s4 (Total Length of All Trajectories in a Hex) saved to 'total_traj_length_s4.csv'")
    
    # s5: Mean Euclidean distance from sub-trajectory centroid to hex centroid
    s5 = mean_sub_traj_centroid_to_hex_centroid_distance(global_hex_multi_df, hex_res1)
    # Save to CSV, dropping zeros or NaNs
    s5_nonzero = s5.dropna(subset=['mean_sub_traj_centroid_distance'])
    s5_nonzero = s5_nonzero[s5_nonzero['mean_sub_traj_centroid_distance'] > 0].copy()
    s5_nonzero = s5_nonzero.sort_values(by='mean_sub_traj_centroid_distance', ascending=False)
    s5_nonzero.to_csv('./csv_files/mean_sub_traj_centroid_s5.csv', index=False)
    print("\nSimilarity Score s5 (Mean Distance from Sub-trajectory Centroid to Hex Centroid) saved to 'mean_sub_traj_centroid_s5.csv'")
    
    # s5b: Mean Euclidean distance from each point to hex centroid
    s5b = mean_point_to_hex_centroid_distance(global_hex_multi_df, hex_res1)
    # Save to CSV, dropping zeros or NaNs
    s5b_nonzero = s5b.dropna(subset=['mean_point_distance'])
    s5b_nonzero = s5b_nonzero[s5b_nonzero['mean_point_distance'] > 0].copy()
    s5b_nonzero = s5b_nonzero.sort_values(by='mean_point_distance', ascending=False)
    s5b_nonzero.to_csv('./csv_files/mean_point_distance_s5b.csv', index=False)
    print("\nSimilarity Score s5b (Mean Distance from Each Point to Hex Centroid) saved to 'mean_point_distance_s5b.csv'")
    
    # s6a and s6b: Water and tree area percentages
    s6a, s6b = area_percentage_in_hex(global_hex_multi_df, hex_res1)
    # Save to CSV, dropping zeros
    s6a_nonzero = s6a[s6a['water_area_%'] > 0].copy()
    s6a_nonzero = s6a_nonzero.sort_values(by='water_area_%', ascending=False)
    s6a_nonzero.to_csv('./csv_files/water_area_pct_s6a.csv', index=False)
    print("\nSimilarity Score s6a (Water Area Percentage) saved to 'water_area_pct_s6a.csv'")
    
    s6b_nonzero = s6b[s6b['tree_area_%'] > 0].copy()
    s6b_nonzero = s6b_nonzero.sort_values(by='tree_area_%', ascending=False)
    s6b_nonzero.to_csv('./csv_files/tree_area_pct_s6b.csv', index=False)
    print("\nSimilarity Score s6b (Tree Area Percentage) saved to 'tree_area_pct_s6b.csv'")
    
    # Return all similarity scores including zero values
    return s1, s2, s3, s4, s5, s5b, s6a, s6b


### 4.4. Plot and save heatmaps for each similarity matrix

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# TODO adjust for s7
def main_similarity_computation(global_hex_multi_df, hex_res1):
    # Compute individual similarity measures
    s1_df, s2_df, s3_df, s4_df, s5_df, s5b_df, s6a_df, s6b_df = compute_and_save_similarity_scores(global_hex_multi_df, hex_res1)
    
    s7_df = pd.read_csv('./csv_files/Amenities_count_along_traj.csv')
    s8_df = pd.read_csv('./csv_files/hex_geohash.csv')

    # Compute similarity matrices
    similarity_matrices = compute_similarity_scores(
        s1_df, s2_df, s3_df, s4_df, s5_df, s5b_df, s6a_df, s6b_df, s7_df, s8_df
    )

    # Compute mean similarity matrix
    mean_similarity_matrix = compute_mean_similarity(similarity_matrices)

    # Mapping of similarity names to descriptions
    sim_descriptions = {
        's1': 'amenity_types',
        's2': 'amen_count',
        's3': 'sub_traj_count',
        's4': 'total_traj_length',
        's5': 'mean_sub_traj_centroid_distance',
        's5b': 'mean_point_distance',
        's6a': 'water_area_pct',
        's6b': 'tree_area_pct',
        's7': 'amen_along_traj',
        's8': 'hex_geohash'
    }
    # Plot and save heatmaps for each similarity matrix
    for sim_name, sim_matrix in similarity_matrices.items():
        plt.figure(figsize=(12, 10))
        sns.heatmap(sim_matrix.astype(float), cmap='viridis')
        plt.title(f'Heatmap of Similarity Matrix {sim_name.upper()}')
        plt.xlabel('Hex IDs')
        plt.ylabel('Hex IDs')
        plt.tight_layout()
        # Use the description for the filename
        description = sim_descriptions.get(sim_name, sim_name)
        plt.savefig(f'./heatmaps/heatmap_{description}_{sim_name}.png')
        plt.close()
        print(f"Heatmap for {sim_name.upper()} saved to 'heatmap_{description}_{sim_name}.png'")

    # Plot and save heatmap for mean similarity matrix
    plt.figure(figsize=(12, 10))
    sns.heatmap(mean_similarity_matrix.astype(float), cmap='viridis')
    plt.title('Heatmap of Mean Similarity Matrix')
    plt.xlabel('Hex IDs')
    plt.ylabel('Hex IDs')
    plt.tight_layout()
    plt.savefig('./heatmaps/heatmap_mean_similarity.png')
    plt.close()
    print("Heatmap for Mean Similarity Matrix saved to 'heatmap_mean_similarity.png'")

    return mean_similarity_matrix, similarity_matrices, s1_df, s2_df, s3_df, s4_df, s5_df, s5b_df, s6a_df, s6b_df, s7_df, s8_df


In [ ]:
mean_similarity_matrix, similarity_matrices, s1_df, s2_df, s3_df, s4_df, s5_df, s5b_df, s6a_df, s6b_df, s7_df, s8_df = main_similarity_computation(global_hex_multi_df, hex_res1)

# 5. Generate large Dataframe to hold all Textual Feature Values of each Hexagon

### 5.1. Merge all dataframes (s1 to s7)

In [ ]:
# Ensure 'amenity_types' in s1_df is a list
def ensure_list(x):
    if isinstance(x, str):
        return x.split(',')
    elif isinstance(x, np.ndarray):
        return x.tolist()
    elif isinstance(x, list):
        return x
    else:
        return []

In [ ]:
# Ensure 'amenity_types' in s1_df is a list using ensure function 
s1_df['amenity_types'] = s1_df['amenity_types'].apply(ensure_list)

# Merge all feature DataFrames
hexagon_features_df = s1_df.merge(s2_df, on='hex_id', how='outer')
hexagon_features_df = hexagon_features_df.merge(s3_df, on='hex_id', how='outer')
hexagon_features_df = hexagon_features_df.merge(s4_df, on='hex_id', how='outer')
hexagon_features_df = hexagon_features_df.merge(s5_df, on='hex_id', how='outer')
hexagon_features_df = hexagon_features_df.merge(s5b_df, on='hex_id', how='outer')
hexagon_features_df = hexagon_features_df.merge(s6a_df, on='hex_id', how='outer')
hexagon_features_df = hexagon_features_df.merge(s6b_df, on='hex_id', how='outer')
hexagon_features_df = hexagon_features_df.merge(s7_df, on='hex_id', how='outer')
hexagon_features_df = hexagon_features_df.merge(s8_df, on='hex_id', how='outer')



# Fill NaN values with default values
hexagon_features_df['amenity_types'] = hexagon_features_df['amenity_types'].apply(lambda x: x if isinstance(x, list) else [])
hexagon_features_df.fillna(0, inplace=True)

len(hexagon_features_df)

### 5.2. Add a description column that describes all features per hexagon for the Sentence Transformer

In [ ]:
def generate_hexagon_description(row):
    # Generate a textual description of a hexagon's features.
    # description: A string summarizing the hexagon's features.
    
    hex_id = row['hex_id']
    amenities = row.get('amenity_types', [])
    amenity_types = ', '.join(amenities) if amenities else 'no amenities'
    amen_count = row.get('amen_count', 0)
    total_amen_along_trajs = row.get('amen_along_trajs', 0)
    sub_traj_count = row.get('sub_traj_count', 0)
    total_traj_length = row.get('total_traj_length', 0)
    mean_sub_traj_centroid_distance = row.get('mean_sub_traj_centroid_distance', 0)
    mean_point_distance = row.get('mean_point_distance', 0)
    water_area_pct = row.get('water_area_%', 0)
    tree_area_pct = row.get('tree_area_%', 0)
    amen_along_traj = row.get('amen_along_traj', 0)
    hex_geohash = row.get('hex_geohash', 0)

    description = (
        f"The hexagon {hex_id} has the following features: "
        f"Amenities include {amenity_types}. "
        f"Total amenity count is {amen_count}. "
        f"Total number of amenities along trajectories is {total_amen_along_trajs}. "
        f"Number of sub-trajectories is {sub_traj_count}. "
        f"Total trajectory length is {total_traj_length:.2f} meters. "
        f"Mean sub-trajectory centroid distance is {mean_sub_traj_centroid_distance:.2f} meters. "
        f"Mean point distance is {mean_point_distance:.2f} meters. "
        f"Water area percentage is {water_area_pct:.2f}%. "
        f"Tree area percentage is {tree_area_pct:.2f}%. "
        f"Amenities along side trajectory is {amen_along_traj}. "
        f"Geohash of the hex is {hex_geohash}."
    )

    return description

# Apply the generate_hexagon_description function to create descriptions.
hexagon_features_df['description'] = hexagon_features_df.apply(generate_hexagon_description, axis=1)

hexagon_features_df.head(2)

In [ ]:
print(hexagon_features_df["description"][0])
print(hexagon_features_df[hexagon_features_df["hex_id"] == "88283082e7fffff"].to_dict("records")[0])  # Using h3_index

# 6. Image Features

### 6.1. Use the OSM API to generate images for each hexagon and display first image

In [ ]:
import matplotlib.pyplot as plt
from io import BytesIO
from PIL import Image
from IPython.display import display
from sklearn.metrics.pairwise import cosine_similarity
import time
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)


def get_hexagon_image(hexagon, width=400, height=400):
    # Get the bounding box of the hexagon
    minx, miny, maxx, maxy = hexagon.bounds
    
    try:
        # Download the street network for the hexagon area
        G = ox.graph_from_bbox(maxy, miny, maxx, minx, network_type='all')
        
        if G.number_of_nodes() == 0:
            # Skip if graph is empty
            print(f"No data for hexagon at ({minx}, {miny}, {maxx}, {maxy})")
            return None
    except Exception as e:
        print(f"Error retrieving graph: {e}")
        return None
    
    # Plot the street network
    fig, ax = ox.plot_graph(
        G,
        show=False,
        close=False,
        edge_color='#999999',
        edge_linewidth=0.5,
        node_size=0,
        figsize=(width / 100, height / 100)
    )
    
    # Remove axis ticks and labels
    ax.set_xticks([])
    ax.set_yticks([])
    ax.axis('off')
    
    # Save the plot to a BytesIO object
    img_buffer = BytesIO()
    plt.savefig(img_buffer, format='png', dpi=100, bbox_inches='tight', pad_inches=0)
    plt.close(fig)
    
    # Convert the BytesIO object to a PIL Image
    img_buffer.seek(0)
    img = Image.open(img_buffer)
    img = img.resize((width, height))
    
    return img

output_dir = 'hexagon_images'
os.makedirs(output_dir, exist_ok=True)

# Generate images for all hexagons
hexagon_images = {}
for idx, row in tqdm(hex_res2.iterrows(), total=len(hex_res2)):
    img = get_hexagon_image(row.geometry)
    if img is not None:
        h3_index = row['h3_index']
        # Use h3_index as the key
        hexagon_images[h3_index] = img
        # Save the image to disk
        img_filename = f'hex_{h3_index}.png'
        img_path = os.path.join(output_dir, img_filename)
        img.save(img_path)
    # time.sleep(1)  # Comply with rate limits and avoid overwhelming the server


print(f"Number of images generated: {len(hexagon_images)}")

# Display the first image
for idx in list(hexagon_images.keys())[:1]:
    print(f"Hexagon index: {idx}")
    display(hexagon_images[idx])

### 6.2. Display all generated images

In [ ]:
import matplotlib.pyplot as plt
def display_traj_image():
    # Display all images
    images = list(hexagon_images.values())
    num_images = len(images)
    print(num_images)
    cols = 5  # Number of images per row
    rows = (num_images + cols - 1) // cols

    fig, axs = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))

    for ax, img in zip(axs.flatten(), images):
        ax.imshow(img)
        ax.axis('off')

    # Hide any unused subplots
    for ax in axs.flatten()[len(images):]:
        ax.axis('off')

    plt.tight_layout()
    plt.show()

'''Turn on for visualisation'''
# display_traj_image()

### 6.3. Compute image similarity scores using the cosine similarity and save its heatmap


In [ ]:
def image_similarity(img1, img2):
    # Convert images to grayscale and flatten
    arr1 = np.array(img1.convert('L')).flatten()
    arr2 = np.array(img2.convert('L')).flatten()
    
    # Calculate cosine similarity
    return cosine_similarity(arr1.reshape(1, -1), arr2.reshape(1, -1))[0][0]

In [ ]:
def plot_save_image_similarity_heatmap(image_similarity_matrix,path):
    # Plot and save the heatmap
    plt.figure(figsize=(14, 14))
    sns.heatmap(image_similarity_matrix, cmap='viridis')
    plt.title('Hexagon Image Similarity Matrix')
    plt.xlabel('Hexagon Index')
    plt.ylabel('Hexagon Index')
    plt.savefig(path)
    plt.close()
    plt.show()

#### 6.3.1. Compute image similarity scores using the cosine similarity and save its heatmap for traj images

In [ ]:
def image_sim_traj_image(hexagon_images):
    image_similarity_scores = {}

    # Get the list of keys from the hexagon_images dictionary
    keys = list(hexagon_images.keys())

    for i in tqdm(range(len(keys))):
        idx_i = keys[i]
        img1 = hexagon_images[idx_i]
        # Set similarity to 1 for the same image
        image_similarity_scores[(idx_i, idx_i)] = 1.0

        for j in range(i + 1, len(keys)):
            idx_j = keys[j]
            img2 = hexagon_images[idx_j]
            
            similarity = image_similarity(img1, img2)
            
            image_similarity_scores[(idx_i, idx_j)] = similarity
            image_similarity_scores[(idx_j, idx_i)] = similarity  # Symmetry

    # Initialize the DataFrame with the correct indices and data type
    image_similarity_matrix = pd.DataFrame(index=keys, columns=keys, dtype=float)

    # Fill in the similarity scores
    for (i, j), score in image_similarity_scores.items():
        image_similarity_matrix.loc[i, j] = score

    # Fill missing values with zeros or appropriate values
    image_similarity_matrix.fillna(0, inplace=True)

    # Optionally, ensure the diagonal is filled with 1s
    np.fill_diagonal(image_similarity_matrix.values, 1)

    # Optionally, adjust display options for Jupyter notebook
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_columns', None)

    # Round the similarity scores for better readability
    similarity_matrix_rounded = image_similarity_matrix.round(2)

    # Save the similarity matrix to a CSV file
    image_similarity_matrix.to_csv('./csv_files/hexagon_image_similarity_matrix.csv')
    plot_save_image_similarity_heatmap(image_similarity_matrix,'./heatmaps/heatmap_image_similarity.png')
    return image_similarity_matrix, similarity_matrix_rounded

#### 6.3.2. Compute image similarity scores using the cosine similarity and save its heatmap for normal images

In [ ]:
def image_similarity_norm(img1, img2):
    # Convert images to grayscale and flatten
    arr1 = img1.flatten()
    arr2 = img2.flatten()
    
    # Calculate cosine similarity
    return cosine_similarity(arr1.reshape(1, -1), arr2.reshape(1, -1))[0][0]

In [ ]:
def image_sim_normal_image():
    hex_ids = pd.read_csv('./csv_files/hex_data.csv')['h3_index']
    hex_imgs = {}
    counter = 0
    for index in hex_ids:
        hex_img = Image.open(f"./hexagon_images_gen/hex_{counter}_{index}_gen.png").convert('L')
        hex_imgs[index] = np.array(hex_img)
        hex_img.close()
        counter += 1
    image_similarity_scores = {}
    # Get the list of keys from the hexagon_images dictionary
    keys = list(hex_imgs.keys())

    for i in tqdm(range(len(keys))):
        idx_i = keys[i]
        img1 = hex_imgs[idx_i]
        # Set similarity to 1 for the same image
        image_similarity_scores[(idx_i, idx_i)] = 1.0

        for j in range(i + 1, len(keys)):
            idx_j = keys[j]
            img2 = hex_imgs[idx_j]
            
            similarity = image_similarity_norm(img1, img2)
            
            image_similarity_scores[(idx_i, idx_j)] = similarity
            image_similarity_scores[(idx_j, idx_i)] = similarity  # Symmetry

    # Initialize the DataFrame with the correct indices and data type
    image_similarity_matrix = pd.DataFrame(index=keys, columns=keys, dtype=float)

    # Fill in the similarity scores
    for (i, j), score in image_similarity_scores.items():
        image_similarity_matrix.loc[i, j] = score

    # Fill missing values with zeros or appropriate values
    image_similarity_matrix.fillna(0, inplace=True)

    # Optionally, ensure the diagonal is filled with 1s
    np.fill_diagonal(image_similarity_matrix.values, 1)

    # Optionally, adjust display options for Jupyter notebook
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_columns', None)
    similarity_matrix_rounded = image_similarity_matrix.round(2)

    # Save the similarity matrix to a CSV file and figure save
    image_similarity_matrix.to_csv('./csv_files/hexagon_normal_image_similarity_matrix.csv')  
    plot_save_image_similarity_heatmap(image_similarity_matrix,'./heatmaps/heatmap_normal_image_similarity.png') 
    return image_similarity_matrix,similarity_matrix_rounded

In [ ]:
normal_image_sim_matrix, normal_image_sim_matrix_rounded = image_sim_normal_image()
traj_image_sim_matrix, traj_image_sim_matrix_rounded = image_sim_traj_image(hexagon_images)

### 6.4. Count the number of pairwise image similarity scores within a certain range.


In [ ]:
'''WARNING OLD CODE'''
# counter = 0
# for (i, j), score in image_similarity_scores.items():
#     if score > 0.4 and score < 1.0:
#         # print(f"Similarity between hexagon {i} and hexagon {j}: {score}")
#         counter += 1
# print(counter)

In [ ]:
hex_res2['geometry']

### 6.5. View maximum mutual pairwise image similarities

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colors
from shapely.geometry import mapping
import numpy as np
import folium
from branca.element import Template, MacroElement
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

# Conditionally execute %matplotlib inline
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except:
    pass  # Not running in a Jupyter environment

# Set 'h3_index' as the index of hex_res2
hex_res2 = hex_res2.set_index('h3_index')

# 1. Get the indices of hexagons that have images
hex_indices_with_images = list(hexagon_images.keys())

# Handle missing indices
missing_indices = set(hex_indices_with_images) - set(hex_res2.index)
if missing_indices:
    print(f"Warning: The following indices are missing in hex_res2 and will be skipped: {missing_indices}")
    hex_indices_with_images = [idx for idx in hex_indices_with_images if idx in hex_res2.index]

hexagons_with_images = hex_res2.loc[hex_indices_with_images].copy()

# 2. Ensure similarity_matrix has the correct indices
if isinstance(traj_image_sim_matrix, np.ndarray):
    if traj_image_sim_matrix.shape != (len(hex_indices_with_images), len(hex_indices_with_images)):
        raise ValueError(
            f"Expected similarity_matrix shape to be ({len(hex_indices_with_images)}, {len(hex_indices_with_images)}), "
            f"but got {traj_image_sim_matrix.shape}"
        )
    mutual_image_similarity_matrix = pd.DataFrame(
        traj_image_sim_matrix,
        index=hex_indices_with_images,
        columns=hex_indices_with_images
    )
elif isinstance(traj_image_sim_matrix, pd.DataFrame):
    # Ensure the indices and columns are h3_index
    mutual_image_similarity_matrix = traj_image_sim_matrix.loc[hex_indices_with_images, hex_indices_with_images]
else:
    raise TypeError("similarity_matrix must be either a NumPy ndarray or a Pandas DataFrame.")

# 3. Identify mutual highest similarity pairs
most_similar = {}

for idx in hex_indices_with_images:
    # Exclude self-similarity
    if idx in mutual_image_similarity_matrix.index:
        mutual_image_similarities = mutual_image_similarity_matrix.loc[idx].drop(labels=[idx], errors='ignore')
        # Find the index with the highest similarity
        if not mutual_image_similarities.empty:
            max_sim_idx = mutual_image_similarities.idxmax()
            max_sim_value = mutual_image_similarities[max_sim_idx]
            most_similar[idx] = (max_sim_idx, max_sim_value)

# Identify mutual pairs
mutual_pairs = []

for idx, (max_sim_idx, _) in most_similar.items():
    if max_sim_idx in most_similar and most_similar[max_sim_idx][0] == idx:
        pair = tuple(sorted((idx, max_sim_idx)))
        if pair not in mutual_pairs:
            mutual_pairs.append(pair)

print(f"Total mutual pairs identified: {len(mutual_pairs)}")

# 4. Assign colors to each mutual pair
def get_N_HexCol(N):
    cmap = plt.get_cmap('tab20')
    colors_list = [colors.rgb2hex(cmap(i % cmap.N)[:3]) for i in range(N)]
    return colors_list

num_pairs = len(mutual_pairs)
colors_list = get_N_HexCol(num_pairs) if num_pairs > 0 else []

# Create a mapping from hexagon index to color
hex_color_map = {}
for i, (idx1, idx2) in enumerate(mutual_pairs):
    color = colors_list[i]
    hex_color_map[idx1] = color
    hex_color_map[idx2] = color

# Assign colors to hexagons
hexagons_with_images['color'] = hexagons_with_images.index.map(lambda idx: hex_color_map.get(idx, 'grey'))

# 5. Print the list of latitudes and longitudes of each Mutual Highest Similarity Pair
# Reproject to a projected CRS for accurate centroid calculation
projected_crs = 'EPSG:3857'
hexagons_with_images_projected = hexagons_with_images.to_crs(projected_crs)
hexagons_with_images_projected['centroid'] = hexagons_with_images_projected.geometry.centroid

# Convert centroids back to WGS84
hexagons_with_images['centroid_wgs84'] = hexagons_with_images_projected['centroid'].to_crs(epsg=4326)

print("List of Latitudes and Longitudes of Mutual Highest Similarity Pairs:")
for idx1, idx2 in mutual_pairs:
    # Get the hexagons
    hex1 = hexagons_with_images.loc[idx1]
    hex2 = hexagons_with_images.loc[idx2]
    
    # Exclude hexagons colored in grey
    if hex1['color'] != 'grey' and hex2['color'] != 'grey':
        # Get centroids in WGS84
        centroid1 = hex1.centroid_wgs84
        centroid2 = hex2.centroid_wgs84
        
        # Get latitudes and longitudes
        lat1, lon1 = centroid1.y, centroid1.x
        lat2, lon2 = centroid2.y, centroid2.x
        
        print(f"Pair: Hexagon {idx1} and Hexagon {idx2}")
        #print(f"Hexagon {idx1}: Latitude {lat1}, Longitude {lon1}")
        #print(f"Hexagon {idx2}: Latitude {lat2}, Longitude {lon2}")
        print("---")

# 6. Create an interactive map with hover functionality
# Reproject to WGS84 for Folium
hexagons_with_images_wgs84 = hexagons_with_images.to_crs(epsg=4326)

# Calculate the overall centroid for map centering
overall_centroid = hexagons_with_images_wgs84.geometry.unary_union.centroid
center_lat, center_lon = overall_centroid.y, overall_centroid.x

# Initialize Folium map
m = folium.Map(location=[center_lat, center_lon], zoom_start=12)

# Prepare GeoJSON data
geojson_data = {
    'type': 'FeatureCollection',
    'features': []
}

for idx, row in hexagons_with_images_wgs84.iterrows():
    # Exclude hexes colored in grey
    if row['color'] != 'grey':
        geojson_feature = {
            'type': 'Feature',
            'properties': {
                'color': row['color'],
                'popup': (
                    f"Hexagon ID: {idx}<br>"
                    f"Latitude: {row['centroid_wgs84'].y:.6f}, Longitude: {row['centroid_wgs84'].x:.6f}"
                )
            },
            'geometry': mapping(row['geometry'])
        }
        geojson_data['features'].append(geojson_feature)

# Define a style function that uses the color property
def style_function(feature):
    return {
        'fillColor': feature['properties']['color'],
        'color': 'black',
        'weight': 1,
        'fillOpacity': 0.6
    }

# Add GeoJSON layer to the map with tooltips
folium.GeoJson(
    data=geojson_data,
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(
        fields=['popup'],
        aliases=[''],
        localize=True
    )
).add_to(m)

# Optional: Add a legend
legend_html = '''
{% macro html() %}
<div style="
    position: fixed; 
    bottom: 50px; left: 50px; width: 200px; height: auto; 
    border:2px solid grey; z-index:9999; font-size:14px;
    background-color:white;
    padding: 10px;
    ">
    <b>Legend</b><br>
    {% for i in range(num_pairs) %}
    &nbsp;<i style="background:{{colors_list[i]}}; width: 10px; height: 10px; display: inline-block;"></i>&nbsp;Pair {{i+1}}<br>
    {% endfor %}
</div>
{% endmacro %}
'''

# Render the template with colors
template = Template(legend_html).render(num_pairs=num_pairs, colors_list=colors_list)

# Create a MacroElement for the legend
macro = MacroElement()
macro._template = Template(template)
m.get_root().html.add_child(macro)

# Display the Folium map
# m


# 7. Compute Embeddings of Textual and Image features in a Dataframe

### 7.1. Generate a larger dataframe containing both textual and image features of Hexagons

In [ ]:
from itertools import combinations
import numpy as np

def create_similarity_dataframe(hex_ids_list, mean_similarity_matrix, traj_image_sim_matrix, normal_image_sim_matrix, approach):
    data = []
    pairs = list(combinations(hex_ids_list, 2))  # NB: order does not matter because they are not permutations

    for hex_id_i, hex_id_j in tqdm(pairs, desc='Creating similarity DataFrame'):
        try:
            # Similarity from S1 to S8
            sim_s1_s8 = mean_similarity_matrix.loc[hex_id_i, hex_id_j]
        except KeyError:
            sim_s1_s8 = np.nan

        try:
            # Traj Image similarity
            sim_images_traj = traj_image_sim_matrix.loc[hex_id_i, hex_id_j]
        except KeyError:
            sim_images_traj = np.nan
        
        try:
            # Noramal Image similarity
            sim_images_normal = normal_image_sim_matrix.loc[hex_id_i, hex_id_j]
        except KeyError:
            sim_images_normal = np.nan

        if approach == 'combined':
            # Combine the similarities (e.g., average), ignoring NaN values
            similarities = [sim_s1_s8, sim_images_traj, sim_images_normal]
            similarities = [sim for sim in similarities if not np.isnan(sim)]
            if similarities:
                similarity_score = np.mean(similarities)
            else:
                similarity_score = np.nan  # Or assign a default value
            data.append({
                'hex_id_1': hex_id_i,
                'hex_id_2': hex_id_j,
                'combined_similarity_score': similarity_score
            })
        elif approach == 'separate':
            data.append({
                'hex_id_1': hex_id_i,
                'hex_id_2': hex_id_j,
                'similarity_s1_s8': sim_s1_s8,
                'similarity_traj_images': sim_images_traj,
                'similarity_normal_images': sim_images_normal

            })

    similarity_df = pd.DataFrame(data)
    return similarity_df

hex_ids_list = list(mean_similarity_matrix.index)

# Check for duplicates in hex_ids_list
if len(hex_ids_list) != len(set(hex_ids_list)):
    print("Warning: Duplicate hex IDs found in hex_ids_list")
    hex_ids_list = list(set(hex_ids_list))
    

similarity_df_combined = create_similarity_dataframe(hex_ids_list, mean_similarity_matrix, traj_image_sim_matrix, normal_image_sim_matrix, approach='combined')
similarity_df_separate = create_similarity_dataframe(hex_ids_list, mean_similarity_matrix, traj_image_sim_matrix, normal_image_sim_matrix, approach='separate')

In [ ]:
similarity_df_combined.head(2)

In [ ]:
similarity_df_separate.head(2)

### 7.2. Generate textual embeddings using Sentence Transformer

In [ ]:
from sentence_transformers import SentenceTransformer

def get_hexagon_textual_embeddings(hexagon_features_df, model_name='all-MiniLM-L6-v2'):
    # Compute textual embeddings for hexagon descriptions.
    model = SentenceTransformer(model_name)
    descriptions = hexagon_features_df['description'].tolist()
    hex_ids = hexagon_features_df['hex_id'].tolist()
    embeddings = model.encode(descriptions, show_progress_bar=True)
    textual_embeddings = dict(zip(hex_ids, embeddings))  # TODO a verification if hex_ids correspond to embeddings
    return textual_embeddings, embeddings


textual_embeddings, embeddings = get_hexagon_textual_embeddings(hexagon_features_df)

### 7.3. Generate image embeddings using a pretrained CNN: ResNet50

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.resnet50 import preprocess_input

def get_image_embeddings(hexagon_images):
    # Extract image embeddings using a pre-trained CNN.

    # Initialize the ResNet50 model
    model = ResNet50(weights='imagenet', include_top=False, pooling='avg')
    image_embeddings = {}
    
    for hex_id, img in tqdm(hexagon_images.items(), desc="Extracting image embeddings"):
        # Resize the image to the required input size
        img = img.resize((224, 224))
        # Convert the image to RGB mode to ensure it has 3 channels
        img = img.convert('RGB')
        x = image.img_to_array(img)
        # Expand dimensions to match the input shape expected by the model
        x = np.expand_dims(x, axis=0)
        # Preprocess the image data
        x = preprocess_input(x)
        # Generate embeddings and set verbose=0 to surpress output progress for tensorflow
        embedding = model.predict(x, verbose=0).flatten()
        # Store the embedding in the dictionary
        image_embeddings[hex_id] = embedding

    return image_embeddings

In [ ]:
def get_normal_image_embeddings():
    hex_ids = pd.read_csv('./csv_files/hex_data.csv')['h3_index']
    hex_imgs = {}
    counter = 0
    for index in hex_ids:
        hex_img = Image.open(f"./hexagon_images_gen/hex_{counter}_{index}_gen.png")
        hex_img.load()
        hex_imgs[index] = hex_img
        counter += 1
    return get_image_embeddings(hex_imgs)

In [ ]:
image_embeddings = get_image_embeddings(hexagon_images)

In [ ]:
normal_image_embeddings = get_normal_image_embeddings()

In [ ]:
# Ensure that h3_index are in the same format
textual_hex_ids = set(textual_embeddings.keys())
image_hex_ids = set(image_embeddings.keys())
normal_image_hex_ids = set(normal_image_embeddings.keys())

print(f"Sample of textual hex IDs: {list(textual_hex_ids)[:5]}")
print(f"Sample of image hex IDs: {list(image_hex_ids)[:5]}")
print(f"Sample of normal image hex IDs: {list(normal_image_hex_ids)[:5]}")


### 7.4. Combine both textual and image embeddings per hexagon

In [ ]:
def combine_embeddings(textual_embeddings, image_embeddings,normal_image_embeddings):
    # Combine textual and image embeddings for each hexagon.
    combined_embeddings = {}
    for hex_id in textual_embeddings.keys():
        text_emb = textual_embeddings[hex_id]
        img_emb = image_embeddings[hex_id]
        norm_img_emb = normal_image_embeddings[hex_id]
        if img_emb is not None:
            combined_emb = np.concatenate([text_emb, img_emb, norm_img_emb])
            combined_embeddings[hex_id] = combined_emb
    return combined_embeddings, text_emb, img_emb, norm_img_emb, combined_emb


combined_embeddings, text_emb, img_emb, norm_img_emb, combined_emb = combine_embeddings(textual_embeddings, image_embeddings,normal_image_embeddings)

print(len(text_emb), len(img_emb), len(norm_img_emb), len(combined_emb))
''' 
all-MiniLM-L6-v2: 384 2048 2432
all-mpnet-base-v2: 768 2048 2816
'''

# 8. Train a Neural Network to predict Combined Embeddings

### 8.1. Generate pairwise dataframes for training and testing sets at hexagon level 

In [ ]:
from itertools import combinations
def create_pairwise_data(hex_ids_subset):
    data = []
    for hex_id_1, hex_id_2 in tqdm(combinations(hex_ids_subset, 2), desc='Creating pairwise data'):
        emb_1 = combined_embeddings[hex_id_1]
        emb_2 = combined_embeddings[hex_id_2]
        # Compute similarity score
        similarity = cosine_similarity([emb_1], [emb_2])[0][0]
        data.append({
            'hex_id_1': hex_id_1,
            'hex_id_2': hex_id_2,
            'embedding_1': emb_1,
            'embedding_2': emb_2,
            'combined_similarity_score': similarity
        })
    pairwise_df = pd.DataFrame(data)
    return pairwise_df

In [ ]:
from sklearn.model_selection import train_test_split

# Get the list of hex IDs
hex_ids = list(combined_embeddings.keys())
hex_ids_pairwise = create_pairwise_data(hex_ids)

# Split the hex IDs into training and testing sets
train_pairwise_df, test_pairwise_df = train_test_split(hex_ids_pairwise, test_size=0.2, random_state=42)

print(f"Number of training pairs: {len(train_pairwise_df)}")
print(f"Number of testing pairs: {len(test_pairwise_df)}")

In [ ]:
test_pairwise_df.head(2)

In [ ]:
train_pairwise_df.head(2)

### 8.2. Prepare training and testing sets

In [ ]:
# Prepare training data
X_train = []
y_train = []

for idx, row in train_pairwise_df.iterrows():
    # Concatenate the embeddings of both hexagons
    feature_vector = np.concatenate([row['embedding_1'], row['embedding_2']])
    X_train.append(feature_vector)
    y_train.append(row['combined_similarity_score'])

X_train = np.array(X_train)
y_train = np.array(y_train)

# Prepare testing data
X_test = []
y_test = []

for idx, row in test_pairwise_df.iterrows():
    feature_vector = np.concatenate([row['embedding_1'], row['embedding_2']])
    X_test.append(feature_vector)
    y_test.append(row['combined_similarity_score'])

X_test = np.array(X_test)
y_test = np.array(y_test)


### 8.3. Train model, compute predictions, and evaluate

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
# Model training
model = MLPRegressor(hidden_layer_sizes=(1024, 512, 256, 128), activation='relu', random_state=42)
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Evaluate efficiency
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Test MAE: {mae:.4f}")
print(f"Test RMSE: {rmse:.4f}")
''' 
With all-mpnet-base-v2:
Test MAE: 0.0383
Test RMSE: 0.0473
With all-MiniLM-L6-v2:
Test MAE: 0.0358
Test RMSE: 0.0456
With New Embeddings:
Test MAE: 0.0621
Test RMSE: 0.0738
With New Embeddings and new data split:
Test MAE: 0.0261
Test RMSE: 0.0327
'''

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import DotProduct, WhiteKernel

kernel = DotProduct() + WhiteKernel()
gpr = GaussianProcessRegressor(kernel=kernel, random_state=42).fit(X_train, y_train)
y_pred_gpr = gpr.predict(X_test)
mae_gpr = mean_absolute_error(y_test, y_pred_gpr)
rmse_gpr = np.sqrt(mean_squared_error(y_test, y_pred_gpr))
print(f"Test MAE: {mae_gpr:.4f}")
print(f"Test RMSE: {rmse_gpr:.4f}")

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error

param_grid = {
    'n_estimators': [200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1],
}

# Initialize the XGBoost regressor
grid_search = GridSearchCV(XGBRegressor(random_state=42), param_grid, cv=3,n_jobs=6,verbose=4)

# Train the model
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)



In [ ]:
# Make predictions
y_pred_xg = grid_search.predict(X_test)
# # Evaluate the model
mae_xg = mean_absolute_error(y_test, y_pred_gpr)
rmse_xg = np.sqrt(mean_squared_error(y_test, y_pred_gpr))
print(f"Test MAE: {mae_xg:.4f}")
print(f"Test RMSE: {rmse_xg:.4f}")
''' 
With New Embeddings:
Test MAE: 0.0728
Test RMSE: 0.0971
With New Embeddings and new data split:
Test MAE: 0.0532
Test RMSE: 0.0696
'''

### 8.4. Visualize Labels vs Predictions using heatmaps

In [ ]:
'''WARNING CODE REQUIRE UPDATE'''

import seaborn as sns
import matplotlib.pyplot as plt
def net_heatmap(test_pairwise_df,model,version = "0",amount = 25):
    # Select 10 hexagons from the test set
    selected_hex_ids = test_pairwise_df[:amount]

    # Create ground truth similarity matrix
    ground_truth_matrix = np.zeros((amount, amount))
    predicted_matrix = np.zeros((amount, amount))

    for i, hex_1 in enumerate(selected_hex_ids.values):
        for j, hex_2 in enumerate(selected_hex_ids.values):
            emb_1 = combined_embeddings[hex_1[0]]
            emb_2 = combined_embeddings[hex_2[1]]
            # Ground truth similarity
            gt_similarity = cosine_similarity([emb_1], [emb_2])[0][0]
            ground_truth_matrix[i, j] = gt_similarity
            
            # Prepare feature vector for prediction
            feature_vector = np.concatenate([emb_1, emb_2]).reshape(1, -1)
            # Predicted similarity
            pred_similarity = model.predict(feature_vector)[0]
            predicted_matrix[i, j] = pred_similarity
            
    # Create a DataFrame for better visualization
    hex_labels_x = selected_hex_ids['hex_id_1']
    hex_labels_y = selected_hex_ids['hex_id_2']
    gt_df = pd.DataFrame(ground_truth_matrix, index=hex_labels_x, columns=hex_labels_y)
    pred_df = pd.DataFrame(predicted_matrix, index=hex_labels_x, columns=hex_labels_y)
    # Plotting ground truth heatmap
    plt.figure(figsize=(12, 12))
    sns.heatmap(gt_df, annot=True, fmt=".2f", cmap='viridis')
    plt.title('Ground Truth Similarity Heatmap')
    plt.savefig(f'./heatmaps/Nets/Ground_truth_similarity_heatmap_{version}')
    plt.close()

    # Plotting predicted heatmap
    plt.figure(figsize=(12, 12))
    sns.heatmap(pred_df, annot=True, fmt=".2f", cmap='viridis')
    plt.title('Predicted Similarity Heatmap')
    plt.savefig(f'./heatmaps/Nets/Predicted_similarity_heatmap_{version}')
    plt.close()


In [ ]:
net_heatmap(test_pairwise_df,model,"MLPReg")

In [ ]:
net_heatmap(test_pairwise_df,gpr,"GaussianProcessReg")

In [ ]:
net_heatmap(test_pairwise_df,grid_search,"XGBReg")

# 9. Exploratory Data Analysis

### 9.1. Descriptive Statistics

In [ ]:
print(hexagon_features_df.describe())

### 9.2. Distribution of amenity counts across hexagons


In [ ]:
# Plot the distribution of amenity counts across hexagons
def plot_amenity_count_distribution(hexagon_features_df):
    plt.figure(figsize=(10, 6))
    sns.histplot(hexagon_features_df['amen_count'], bins=30, kde=True)
    plt.title('Distribution of Amenity Counts per Hexagon')
    plt.xlabel('Amenity Count')
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.savefig('./EDA/Amenity_counts_distribution.png')
    plt.close()


plot_amenity_count_distribution(hexagon_features_df)

### 9.3. Distribution of Amenities along trajectories

In [ ]:
# Plot distribution of Amenities along trajectory
def plot_amenity_count_along_trajs_distribution(hexagon_features_df):
    plt.figure(figsize=(10, 6))
    sns.histplot(hexagon_features_df['amen_along_trajs'], bins=30, kde=True)  # set kde=True for line
    plt.title("Distribution of Amenity Count along Trajectories")
    plt.xlabel("Amenity Count")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.savefig("./EDA/Amenity_counts_along_trajs_distribution.png")
    plt.close()


plot_amenity_count_along_trajs_distribution(hexagon_features_df)

### 9.4. Most frequent amenities across all hexagons

In [ ]:
# Plot the most common amenities across all hexagon
def plot_most_common_amenities(hexagon_features_df, top_n):
    # Explode the amenity types without splitting
    s1_df_exploded = hexagon_features_df.copy()
    # Ensure that 'amenity_types' is a list; if it's an ndarray, convert it to a list
    s1_df_exploded['amenity_types'] = s1_df_exploded['amenity_types'].apply(lambda x: x.tolist() if isinstance(x, np.ndarray) else x)
    amenities_series = s1_df_exploded.explode('amenity_types')['amenity_types']
    # Drop any NaN values that may result from empty lists
    amenities_series = amenities_series.dropna()
    # Count the occurrences
    amenity_counts = amenities_series.value_counts().head(top_n)
   
    plt.figure(figsize=(10, 6))
    sns.barplot(x=amenity_counts.values, y=amenity_counts.index)
    plt.title(f'Top {top_n} Frequent Amenities')
    plt.xlabel('Count')
    plt.ylabel('Amenity Type')
    plt.tight_layout()
    plt.savefig('./EDA/Amenity_types_most_frequent.png')
    plt.close()


plot_most_common_amenities(hexagon_features_df, 20)

### 9.5. Sub-Trajectories per hexagon

In [ ]:
from shapely.geometry import LineString

# Plot sub-trajectories on a map.
def plot_sub_trajectories(global_hex_multi_df):
    # Convert MultiPoints to LineStrings for plotting
    trajectories = []
    for idx, row in global_hex_multi_df.iterrows():
        points = [Point(point.x, point.y) for point in row['Multi_point'].geoms]
        if len(points) >= 2:
            line = LineString(points)
            trajectories.append({'geometry': line})
    trajectories_gdf = gpd.GeoDataFrame(trajectories, crs='EPSG:4326')
    fig, ax = plt.subplots(figsize=(10, 10))
    trajectories_gdf.plot(ax=ax, linewidth=1, color='blue')
    plt.title('Sub-Trajectories')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.tight_layout()
    plt.savefig('./EDA/Sub_trajectories.png')
    plt.close()


plot_sub_trajectories(global_hex_multi_df)

### 9.6. Distribution of total trajectory lengths

In [ ]:
#  Plot the distribution of total trajectory lengths.
def plot_trajectory_length_distribution(hexagon_features_df):
    plt.figure(figsize=(10, 6))
    sns.histplot(hexagon_features_df['total_traj_length'], bins=30, kde=True)
    plt.title('Distribution of Total Trajectory Lengths per Hexagon')
    plt.xlabel('Total Trajectory Length (meters)')
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.savefig('./EDA/Total_sub_traj_length.png')
    plt.close()

plot_trajectory_length_distribution(hexagon_features_df)

### 9.7. Map display for an attribute

In [ ]:
#  Plot hexagons colored by an attribute e.g total_traj_length
def plot_hexagon_attribute_map(hex_res1, attribute_df, attribute_name, title):
    # Merge the attribute data with the hexagon GeoDataFrame
    hex_gdf = hex_res1.merge(attribute_df, left_on='h3_index', right_on='hex_id', how='left')
    fig, ax = plt.subplots(figsize=(12, 10))
    hex_gdf.plot(column=attribute_name, ax=ax, legend=True, cmap='viridis', edgecolor='black')
    plt.title(title)
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.tight_layout()
    plt.savefig('./EDA/Hexagon_map_total_traj_length.png')
    plt.close()


plot_hexagon_attribute_map(hex_res1, hexagon_features_df, 'total_traj_length', 'Total Trajectory Length per Hexagon')

100 meter radius

### 9.8. Correlation matrix of numerical features s1 to s7

In [ ]:
# Plot correlation matrix of numerical features
def plot_correlation_matrix_pairplot(hexagon_features_df, numerical_features):
    plt.figure(figsize=(19, 19))
    corr_matrix = hexagon_features_df[numerical_features].corr()
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
    plt.title('Correlation Matrix of Numerical Features')
    plt.savefig('./EDA/Correlation_matrix.png')
    plt.close()

    # Pairplot
    sns.pairplot(hexagon_features_df[numerical_features])
    plt.title('Pairplot of all Numerical Features')
    plt.savefig('./EDA/Pairplot.png')
    plt.close()


numerical_features = ['amen_count', 'sub_traj_count', 'total_traj_length',
                      'mean_sub_traj_centroid_distance', 'mean_point_distance',
                      'water_area_%', 'tree_area_%', 'amen_along_trajs']
plot_correlation_matrix_pairplot(hexagon_features_df, numerical_features)

### 9.9. Word Cloud for Amenity Types

In [ ]:
from wordcloud import WordCloud

# Word Cloud for Amenity Types
def plot_wordcloud_amenity_types(hexagon_features_df):

    amenity_text = ' '.join(hexagon_features_df['amenity_types'].explode().dropna().astype(str))
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate(amenity_text)

    plt.figure(figsize=(15, 7.5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title('Word Cloud of Amenity Types')
    plt.savefig('./EDA/Wordcloud_amenity_types.png')
    plt.close()

plot_wordcloud_amenity_types(hexagon_features_df)

### 9.10 Word Cloud for Amenities along Trajectories

In [ ]:
from ast import literal_eval

# Read the CSV file correctly
amenities_df = pd.read_csv('./csv_files/Amenities_count_along_traj_s7.csv')

# Initialize dictionary to store total counts
amenity_total_counts = {}

# Iterate over the rows and aggregate counts
for idx, row in amenities_df.iterrows():
    amenity_counts_str = row['amen_count_traj']
    # Convert string representation of list of tuples to an actual list
    try:
        amenity_counts_list = literal_eval(amenity_counts_str)
        for amenity, count in amenity_counts_list:
            # Add counts to the dictionary
            amenity_total_counts[amenity] = amenity_total_counts.get(amenity, 0) + count
    except Exception as e:
        print(f"Error parsing row {idx}: {e}")
        print(f"amenity_counts_str: {amenity_counts_str}")
        continue  # Skip rows that cause errors

# Generate the word cloud using frequencies
wordcloud = WordCloud(width=800, height=400, background_color='white')
wordcloud.generate_from_frequencies(amenity_total_counts)

plt.figure(figsize=(15, 7.5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('WordCloud of Amenities along Trajectories')
plt.savefig('./EDA/Wordcloud_amenities_along_trajs.png')
plt.close()


In [ ]:
#TODO share images of the 5 most similar hex images and 5 least similar hex images based on image similarity only and based on text similarity only